# ЛР-02: Запчасти на ремонтные базы

## Student notebook: military 02

Этот notebook предназначен для самостоятельного решения.

Готового оптимального плана и заполненного решателя здесь нет.

## 1. Зачем нужен этот кейс

На ремонтные базы нужно доставить больше комплектов, чем сейчас есть на складах.

После этого notebook-а студент должен уметь:

1. замечать дефицит снабжения до запуска solver-а;
2. вводить фиктивного поставщика для балансировки;
3. фиксировать смысл неудовлетворённого спроса в выводе;
4. аккуратно интерпретировать полученную матрицу перевозок.

## 2. Исходные данные

Тип задачи: **открытая, спрос больше, чем запасов**.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Склад A | 25 |
| Склад B | 30 |
| Склад C | 20 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Рембаза 1 | 15 |
| Рембаза 2 | 20 |
| Рембаза 3 | 18 |
| Рембаза 4 | 30 |

### Матрица затрат

| Откуда / Куда | Рембаза 1 | Рембаза 2 | Рембаза 3 | Рембаза 4 |
| --- | --- | --- | --- | --- |
| Склад A | 7 | 5 | 9 | 11 |
| Склад B | 6 | 4 | 7 | 8 |
| Склад C | 8 | 6 | 5 | 7 |

## 3. Что нужно сделать

Идите тем же маршрутом, что и в теории ЛР-02:

1. Проверьте баланс запасов и спроса.
2. Если задача открытая, добавьте фиктивный узел и поясните его смысл.
3. Запишите математическую модель через переменные $x_{ij}$.
4. Сформируйте `c`, `A_eq`, `b_eq`, `bounds` для `linprog`.
5. Проверьте, что `A_eq x = b_eq` действительно задаёт строки и столбцы.
6. После собственной попытки решите задачу через Python.
7. Кратко объясните реальные и фиктивные маршруты.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9

# Шаг 1: записываем данные задачи, не меняя числовую постановку.
supplier_names = [
    'Склад A',
    'Склад B',
    'Склад C',
]
consumer_names = [
    'Рембаза 1',
    'Рембаза 2',
    'Рембаза 3',
    'Рембаза 4',
]

supplies = np.array(
    [
        25,
        30,
        20,
    ],
    dtype=float,
)

demands = np.array(
    [
        15,
        20,
        18,
        30,
    ],
    dtype=float,
)

costs = np.array(
    [
        [7, 5, 9, 11],
        [6, 4, 7, 8],
        [8, 6, 5, 7],
    ],
    dtype=float,
)

# Шаг 2: выводим векторы и матрицу затрат в читаемом виде.
supply_df = pd.DataFrame({"запас": supplies}, index=supplier_names)
demand_df = pd.DataFrame({"спрос": demands}, index=consumer_names)
cost_df = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df)

print("sum supply =", supplies.sum())
print("sum demand =", demands.sum())


## 4. Шаблон для самостоятельной сборки модели

Ниже намеренно оставлены `TODO`. Это не готовое решение, а guided skeleton:
он показывает правильные имена, порядок шагов и форму LP-модели, но ключевые
строки вы заполняете самостоятельно.


In [ ]:
def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Готовит закрытую транспортную модель через фиктивный узел.

    Аргументы:
        supplies (np.ndarray): Вектор запасов ``a_i``.
        demands (np.ndarray): Вектор спроса ``b_j``.
        costs (np.ndarray): Матрица затрат ``c_ij``.
        supplier_names (list[str]): Названия строк поставщиков.
        consumer_names (list[str]): Названия столбцов потребителей.
        dummy_cost (float): Стоимость фиктивной строки или столбца.

    Возвращает:
        tuple: Сбалансированные векторы, матрица, подписи и пояснение.
    """

    # Шаг 1: работаем с копиями, чтобы исходные данные задачи не менялись.
    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    # TODO: если balance_difference > 0, добавьте фиктивного потребителя.
    # Подсказка: используйте np.append(...) и np.column_stack(...).

    # TODO: если balance_difference < 0, добавьте фиктивного поставщика.
    # Подсказка: используйте np.append(...) и np.vstack(...).

    # TODO: замените это пояснение после обработки всех случаев баланса.
    balance_note = "TODO: объясните, закрытая задача или нужен фиктивный узел."

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )


def build_transport_lp(supplies, demands, costs):
    """Собирает ``c``, ``A_eq``, ``b_eq`` и ``bounds`` для ``linprog``.

    Аргументы:
        supplies (np.ndarray): Сбалансированный вектор запасов.
        demands (np.ndarray): Сбалансированный вектор спроса.
        costs (np.ndarray): Сбалансированная матрица затрат.

    Возвращает:
        dict[str, object]: Части LP-модели в канонической форме.
    """

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    # TODO: вектор цели должен идти в том же порядке, что и costs.flatten().
    c = None

    # TODO: соберите строки A_eq сначала для поставщиков, потом для потребителей.
    A_eq = None

    # TODO: b_eq должен совпадать с порядком строк матрицы A_eq.
    b_eq = None

    # TODO: каждая переменная x_ij неотрицательна, значит bound равен (0.0, None).
    bounds = None

    # Подсказка: используйте эту формулу индекса внутри обоих циклов.
    route_index = "supplier_idx * consumer_count + consumer_idx"

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "route_index_hint": route_index,
        "variable_count": variable_count,
    }


# Шаг 1: сбалансируйте задачу после заполнения TODO выше.
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("balanced supply =", balanced_supplies.sum())
print("balanced demand =", balanced_demands.sum())

# TODO: раскомментируйте проверку после реализации балансировки.
# assert np.allclose(balanced_supplies.sum(), balanced_demands.sum())

# Шаг 2: соберите LP-модель после заполнения build_transport_lp(...).
lp_model = build_transport_lp(balanced_supplies, balanced_demands, balanced_costs)

# TODO: когда c, A_eq, b_eq и bounds готовы, выведите их как таблицы.
# route_labels = [
#     f"x_{supplier_idx + 1},{consumer_idx + 1}"
#     for supplier_idx in range(len(balanced_supplier_names))
#     for consumer_idx in range(len(balanced_consumer_names))
# ]
# display(pd.DataFrame({"переменная": route_labels, "стоимость c": lp_model["c"]}))
# display(pd.DataFrame(lp_model["A_eq"], columns=route_labels))
# display(pd.DataFrame({"b_eq": lp_model["b_eq"]}))

# Шаг 3: запускайте solver только после полной сборки LP-модели.
# result = linprog(
#     lp_model["c"],
#     A_eq=lp_model["A_eq"],
#     b_eq=lp_model["b_eq"],
#     bounds=lp_model["bounds"],
#     method="highs",
# )
# assert result.success, result.message
# plan = result.x.reshape(len(balanced_supplier_names), len(balanced_consumer_names))
# plan_df = pd.DataFrame(plan, index=balanced_supplier_names, columns=balanced_consumer_names)
# display(plan_df)

# Шаг 4: после решения обязательно проверьте суммы по строкам и столбцам.
# assert np.allclose(plan_df.sum(axis=1), balanced_supplies)
# assert np.allclose(plan_df.sum(axis=0), balanced_demands)


## 5. Что должно быть в отчёте

1. Таблица исходных данных и проверка баланса.
2. Полная математическая постановка через $x_{ij}$.
3. Пояснение, нужен ли фиктивный поставщик или фиктивный потребитель.
4. Векторно-матричная запись `c`, `A_eq`, `b_eq`, `bounds`.
5. Проверка через `linprog`.
6. Таблица оптимального плана перевозок.
7. Проверка сумм по строкам и столбцам.
8. Содержательная интерпретация ненулевых маршрутов.

## 6. Контрольный чек-лист

- [ ] Я сам проверил баланс до запуска solver-а.
- [ ] Я осмысленно собрал `A_eq` и `b_eq` через индекс `route_index`.
- [ ] Я проверил `bounds` и неотрицательность всех $x_{ij}$.
- [ ] Я отличаю реальные маршруты от фиктивного узла.
- [ ] Я объяснил полученный план простыми словами.
